In [1]:
import pandas as pd
import json, os, ast, math
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
import numpy as np
from collections import Counter

In [2]:
# 利用DTW计算曲线相似性
def path1(curve1,curve2):
    if (max(curve1)-min(curve1))>0:
        curve1 = [(i-min(curve1))/(max(curve1)-min(curve1)) for i in curve1]
    if (max(curve2)-min(curve2))>0:
        curve2 = [(i-min(curve2))/(max(curve2)-min(curve2)) for i in curve2]

    # 定义距离函数，这里使用欧氏距离
    distance = lambda x, y: euclidean(x, y)

    # 使用fastdtw计算DTW距离
    distance, path = fastdtw(curve1, curve2, dist=distance)
    return path

In [3]:
df = pd.read_excel("Data of collaboration trajectories.xlsx")
df.head()

,机构,姓名,年份,年龄,广度,标准化广度,深度,标准化深度,C指数,标准化C指数,研究领域,学科
0,北京邮电大学,杨义先,1994,4,1,0.000000,1,0.0,1,0.0,信号与信息处理,工程
1,北京邮电大学,杨义先,1995,5,1,0.000000,1,0.0,1,0.0,信号与信息处理,工程
2,北京邮电大学,杨义先,1996,6,1,0.000000,1,0.0,1,0.0,信号与信息处理,工程
3,北京邮电大学,杨义先,1997,7,1,0.000000,1,0.0,1,0.0,信号与信息处理,工程
4,北京邮电大学,杨义先,1998,8,3,0.005222,1,0.0,1,0.0,信号与信息处理,工程


In [4]:
nameList = [f"{df.iloc[i,0]}__{df.iloc[i,1]}" for i in range(df.shape[0])]
nameList = list(set(nameList))

In [5]:
name1 = "上海交通大学__孙宝德"
list3 = []
curve1 = df[(df['姓名']==name1.split('__')[1])&(df['机构']==name1.split('__')[0])]['广度'].tolist()
for name2 in nameList:
    df00 = df[(df['姓名']==name2.split('__')[1])&(df['机构']==name2.split('__')[0])]
    df00.index= [j for j in range(len(df00.index))]
    if name2 == name1:
        for i in range(df00.shape[0]):
            list3.append([name2,df00.iloc[i,0],df00.iloc[i,1],df00.iloc[i,2],
                         df00.iloc[i,3],df00.iloc[i,4],df00.iloc[i,5],df00.iloc[i,10],
                         df00.iloc[i,11],df00.iloc[i,3]])
    else:
        curve2 = df00['广度'].tolist()
        path = path1(curve1,curve2)
        for i in range(len(curve1)):
            yearlist = [(item[1]+4) for item in path if item[0]==i]
            df000 = df00[df00['年龄'].isin(yearlist)]
            df000.index= [j for j in range(len(df000.index))]
            list3.append([name2,df000.iloc[0,0],df000.iloc[0,1],int(np.mean(df000['年份'].tolist())),
                         int(np.mean(df000['年龄'].tolist())),np.mean(df000['广度'].tolist()),
                         np.mean(df000['标准化广度'].tolist()),
                         df000.iloc[0,10],df000.iloc[0,11],i+4])

In [6]:
df3 = pd.DataFrame(list3,columns=['机构__姓名','机构','姓名','年份','年龄','广度',
                                 '标准化广度','研究领域','学科','校正年份'])
df3.head()

,机构__姓名,机构,姓名,年份,年龄,广度,标准化广度,研究领域,学科,校正年份
0,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1985,4,2.000000,0.000000,航空宇航推进理论与工程,工程,4
1,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1987,6,4.666667,0.030651,航空宇航推进理论与工程,工程,5
2,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1989,8,6.000000,0.045977,航空宇航推进理论与工程,工程,6
3,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1990,9,6.000000,0.045977,航空宇航推进理论与工程,工程,7
4,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1993,12,7.000000,0.057471,航空宇航推进理论与工程,工程,8


In [7]:
df3.to_excel(f'学者合作广度成长轨迹聚类分析数据.xlsx',index=False)

In [8]:
name1 = "上海交通大学__孙宝德"
list3 = []
curve1 = df[(df['姓名']==name1.split('__')[1])&(df['机构']==name1.split('__')[0])]['深度'].tolist()
for name2 in nameList:
    df00 = df[(df['姓名']==name2.split('__')[1])&(df['机构']==name2.split('__')[0])]
    df00.index= [j for j in range(len(df00.index))]
    if name2 == name1:
        for i in range(df00.shape[0]):
            list3.append([name2,df00.iloc[i,0],df00.iloc[i,1],df00.iloc[i,2],
                         df00.iloc[i,3],df00.iloc[i,4],df00.iloc[i,5],df00.iloc[i,10],
                         df00.iloc[i,11],df00.iloc[i,3]])
    else:
        curve2 = df00['深度'].tolist()
        path = path1(curve1,curve2)
        for i in range(len(curve1)):
            yearlist = [(item[1]+4) for item in path if item[0]==i]
            df000 = df00[df00['年龄'].isin(yearlist)]
            df000.index= [j for j in range(len(df000.index))]
            list3.append([name2,df000.iloc[0,0],df000.iloc[0,1],int(np.mean(df000['年份'].tolist())),
                         int(np.mean(df000['年龄'].tolist())),np.mean(df000['深度'].tolist()),
                         np.mean(df000['标准化深度'].tolist()),
                         df000.iloc[0,10],df000.iloc[0,11],i+4])

In [9]:
df3 = pd.DataFrame(list3,columns=['机构__姓名','机构','姓名','年份','年龄','深度',
                                 '标准化深度','研究领域','学科','校正年份'])
df3.head()

,机构__姓名,机构,姓名,年份,年龄,深度,标准化深度,研究领域,学科,校正年份
0,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1985,4,1.000000,0.000000,航空宇航推进理论与工程,工程,4
1,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1988,7,2.666667,0.052083,航空宇航推进理论与工程,工程,5
2,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1993,12,1.000000,0.000000,航空宇航推进理论与工程,工程,6
3,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1994,13,1.000000,0.000000,航空宇航推进理论与工程,工程,7
4,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1997,16,2.833333,0.057292,航空宇航推进理论与工程,工程,8


In [10]:
df3.to_excel(f'学者合作深度成长轨迹聚类分析数据.xlsx',index=False)

In [11]:
name1 = "上海交通大学__孙宝德"
list3 = []
curve1 = df[(df['姓名']==name1.split('__')[1])&(df['机构']==name1.split('__')[0])]['C指数'].tolist()
for name2 in nameList:
    df00 = df[(df['姓名']==name2.split('__')[1])&(df['机构']==name2.split('__')[0])]
    df00.index= [j for j in range(len(df00.index))]
    if name2 == name1:
        for i in range(df00.shape[0]):
            list3.append([name2,df00.iloc[i,0],df00.iloc[i,1],df00.iloc[i,2],
                         df00.iloc[i,3],df00.iloc[i,4],df00.iloc[i,5],df00.iloc[i,10],
                         df00.iloc[i,11],df00.iloc[i,3]])
    else:
        curve2 = df00['C指数'].tolist()
        path = path1(curve1,curve2)
        for i in range(len(curve1)):
            yearlist = [(item[1]+4) for item in path if item[0]==i]
            df000 = df00[df00['年龄'].isin(yearlist)]
            df000.index= [j for j in range(len(df000.index))]
            list3.append([name2,df000.iloc[0,0],df000.iloc[0,1],int(np.mean(df000['年份'].tolist())),
                         int(np.mean(df000['年龄'].tolist())),np.mean(df000['C指数'].tolist()),
                         np.mean(df000['标准化C指数'].tolist()),
                         df000.iloc[0,10],df000.iloc[0,11],i+4])

In [12]:
df3 = pd.DataFrame(list3,columns=['机构__姓名','机构','姓名','年份','年龄','C指数',
                                 '标准化C指数','研究领域','学科','校正年份'])
df3.head()

,机构__姓名,机构,姓名,年份,年龄,C指数,标准化C指数,研究领域,学科,校正年份
0,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1985,4,1.00,0.000000,航空宇航推进理论与工程,工程,4
1,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1988,7,2.00,0.142857,航空宇航推进理论与工程,工程,5
2,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1993,12,1.00,0.000000,航空宇航推进理论与工程,工程,6
3,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1994,13,1.00,0.000000,航空宇航推进理论与工程,工程,7
4,北京航空航天大学__杨威迦,北京航空航天大学,杨威迦,1998,17,2.75,0.250000,航空宇航推进理论与工程,工程,8


In [13]:
df3.to_excel(f'学者C指数成长轨迹聚类分析数据.xlsx',index=False)